# TP · Identidad y emoción de actores argentinos — Grupo 6 (versión final)
**IAyAA II · Licenciatura en Ciencia de Datos · UCA Rosario · 2026**

Martín Masuelli · Francisco Miles · Tomás Lizazu · Bautista Verna · Juan Diego Ancarani · Manuel Lafosse

Dos tareas sobre los recortes de rostro del curso: **identidad** (20 actores) y **emoción** (alegría, enojo, neutral, sorpresa, tristeza).

Este notebook contiene solo lo que se entrega: **la CNN propia más efectiva** (la que compite, entrenada desde cero) y **ResNet50** como referencia de transferencia. Todo el camino recorrido (CNN v1, búsqueda con KerasTuner, versiones anteriores, 16 experimentos) está en el notebook completo `notebooks/TP2_Grupo6.ipynb` del repositorio, en `docs/EXPERIMENTOS_CNN.md` y en la bitácora `docs/BITACORA.md`.

| # | Sección |
|---|---|
| 1 | Entorno y datos (Drive del curso en solo lectura) |
| 2 | Registro del dataset y recuperación del video de origen |
| 3 | Partición **por video** (sin fugas) |
| 4 | Baselines |
| 5 | Pipeline y herramientas de entrenamiento |
| 6 | **CNN propia final** (una red por tarea) |
| 7 | ResNet50 (transferencia, referencia) |
| 8 | Comparación en validación |
| 9 | Test (una sola vez) |
| 10 | ¿El modelo mira la cara? (oclusión) |
| 11 | Predicción sobre fotos nuevas |
| 12 | Conclusiones |

**Requisitos:** GPU T4 y, en la raíz de "Mi unidad", un acceso directo a `CARPETA COMPARTIDA PARA TODOS` (y, si la cuenta no ve las imágenes ahí, otro a `Imagenes_Finales`). Tiempo total ≈ 45 min.

## 1. Entorno y datos
Toda constante que afecta un resultado está en la celda 1.1. Los hiperparámetros de la CNN son los que eligió **KerasTuner** (NB09) en la búsqueda del notebook completo (experimento E16); acá se usan fijos para no repetir una búsqueda de 1,5 h.

In [ ]:
# 1.1 Librerías, configuración y semillas
import os, re, glob, time, random, shutil, hashlib, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from IPython.display import display

import tensorflow as tf
import keras
from keras import layers
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
EMOCIONES = ("alegria", "enojo", "neutral", "sorpresa", "tristeza")
TAREAS = ("actor", "emocion")
FRACCIONES = {"train": 0.70, "val": 0.15, "test": 0.15}
CSV_MASTER, CARPETA_IMAGENES = "etiquetas_master_final.csv", "Imagenes_Finales"
IMG_SIZE = 224             # recortes del curso = entrada nativa de ResNet50
IMG_SIZE_BASELINE = 64     # baselines clásicos sobre píxeles en gris

# CNN propia final: hiperparámetros elegidos por KerasTuner por accuracy de validación (E16, D-018/D-019)
IMG_SIZE_CNN, BATCH_CNN = 112, 64
HP_CNN = {"base": 48, "dropout": 0.4, "l2": 0.0, "lr": 1.4e-3, "rotacion": 0.04, "zoom": 0.10, "color": 0.10}
EPOCAS_MAX_CNN, PACIENCIA_CNN = 150, 15

# ResNet50: fase 1 cabeza (lr 1e-3), fase 2 descongelando conv4 + conv5 (lr 1e-5), elegido en validación (E15)
BATCH_RESNET = 32
RESNET_PRIMERA_DESCONGELADA = "conv4_block1_1_conv"

# Identidades que el CSV del curso escribe distinto
MAPA_ACTORES = {
    "tini": "tini_stoessel", "georginabarbarossa": "georgina_barbarossa",
    "oscarmartinez": "oscar_martinez", "juliochavez": "julio_chavez",
    "pachupena": "pachu_pena", "rodrigodelaserna": "rodrigo_de_la_serna",
    "francella": "guillermo_francella", "darin": "ricardo_darin", "lali": "lali_esposito",
    "milei": "javier_milei", "barassi": "dario_barassi", "momo_benavides": "momo",
    "geronimo_benavides": "momo", "flavio_azzaro": "azzaro",
    "franco_colapinto": "colapinto", "lionel_scaloni": "scaloni",
}


def fijar_semillas(seed=SEED):
    # Fija todas las fuentes de azar; se llama al inicio y antes de construir cada modelo.
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


fijar_semillas()
sns.set_theme(style="whitegrid")
print(f"TensorFlow {tf.__version__} · Keras {keras.__version__}")
print("GPU:", tf.config.list_physical_devices("GPU") or "NO — activar GPU T4 en Entorno de ejecución")

In [ ]:
# 1.2 Drive del curso en SOLO LECTURA + copia local de trabajo
from google.colab import drive
drive.mount("/content/drive", readonly=True)   # readonly: es imposible modificar el Drive del curso

candidatas = (glob.glob("/content/drive/MyDrive/CARPETA COMPARTIDA PARA TODOS")
              + glob.glob("/content/drive/MyDrive/*/CARPETA COMPARTIDA PARA TODOS"))
assert candidatas, "Falta el acceso directo a 'CARPETA COMPARTIDA PARA TODOS' en Mi unidad."
CARPETA_CURSO = Path(candidatas[0])
# Si una cuenta no ve las imágenes dentro del dataset del curso (permisos), sirve un acceso directo a la carpeta de imágenes.
origen_imgs = next((p for p in (CARPETA_CURSO / "DATASET_FINAL" / CARPETA_IMAGENES,
                                Path("/content/drive/MyDrive") / CARPETA_IMAGENES,
                                Path("/content/drive/MyDrive/DATASET_FINAL") / CARPETA_IMAGENES) if p.exists()), None)
assert origen_imgs, f"Esta cuenta no ve '{CARPETA_IMAGENES}': pedir acceso de lector o agregar un acceso directo en Mi unidad."
DATASET = Path("/content/datos/DATASET_FINAL")
if not (DATASET / CARPETA_IMAGENES).exists():      # copia local: más rápida y aísla el original
    shutil.rmtree(DATASET, ignore_errors=True)       # descarta una copia incompleta (disco de Colab, no el Drive)
    shutil.copytree(CARPETA_CURSO / "DATASET_FINAL", DATASET, ignore=shutil.ignore_patterns("Metricas_Dataset", CARPETA_IMAGENES))
    shutil.copytree(origen_imgs, DATASET / CARPETA_IMAGENES)
print("Imágenes tomadas de:", origen_imgs)
print("Copia de trabajo:", DATASET, "→", len(os.listdir(DATASET / CARPETA_IMAGENES)), "imágenes")

## 2. Registro del dataset
El CSV maestro del curso tiene tres problemas que el código resuelve: nombres con tildes y la `ñ` codificada como `#U00f1` (se cruzan por una *clave* normalizada), identidades escritas distinto (`tini` / `tini_stoessel`, se unifican con `MAPA_ACTORES`) y, sobre todo, **no tiene el video de origen**, indispensable para partir sin fugas. El video se recupera de los registros que subió cada grupo (solo lectura) y, si una imagen no figura, del patrón de su nombre.

In [ ]:
# 2.1 Normalización, video de origen y registro (una fila por imagen utilizable)
def clave_archivo(nombre):
    # 'PachuPe#U00f1a_Alegr#U00eda.jpg' y 'PachuPeña_Alegría.jpg' -> 'pachupena_alegria.jpg'
    s = re.sub(r"#U([0-9a-fA-F]{4})", lambda m: chr(int(m.group(1), 16)), str(nombre))
    s = unicodedata.normalize("NFKD", s)
    return "".join(c for c in s if not unicodedata.combining(c)).lower().strip()


def normalizar_etiqueta(texto):
    s = re.sub(r"[\s\-_]+", "_", clave_archivo(texto))
    return re.sub(r"[^a-z0-9_]", "", s).strip("_")


def normalizar_actor(nombre):
    actor = normalizar_etiqueta(nombre)
    return MAPA_ACTORES.get(actor, actor)


PATRONES_VIDEO = (
    re.compile(r"^.+?__(?P<video>.+?)__f\d+\.jpe?g$"),          # G1/G10: actor__video__f0000123.jpg
    re.compile(r"^(?P<video>.+?)_frame_\d+\.jpe?g$"),           # G4: Cande1_frame_000010.jpg
    re.compile(r"^(?P<video>.+?)(?:_p\d+)?_\d+\.jpe?g$"),       # G7: momo_video_1_p2_0003.jpg
)


def video_desde_nombre(clave):
    for patron in PATRONES_VIDEO:
        m = patron.match(clave)
        if m:
            return m.group("video")
    return None


def leer_registros_grupos(carpeta_curso):
    # Mapa clave -> video a partir del registro por imagen de cada grupo (solo lectura).
    fuentes, cols = [], ["archivo", "actor", "emocion", "video", "escena"]
    for g in ("G3", "G7", "G8"):
        p = carpeta_curso / g / "etiquetas_master.txt"
        if p.exists():
            fuentes.append(pd.read_csv(p, sep=";", header=None, names=cols)[["archivo", "video"]])
    especiales = {"G1": ("registro_grupo.csv", "archivo", "video", {}),
                  "G4": ("labels_224_final_grupo4.csv", "imagen", "video", {}),
                  "G6": ("DATASET_FINAL.csv", "imagen", "video_origen", {"encoding": "utf-8-sig"})}
    for g, (nombre, c_arch, c_vid, kw) in especiales.items():
        p = carpeta_curso / g / nombre
        if p.exists():
            d = pd.read_csv(p, **kw).dropna(subset=[c_arch, c_vid])
            fuentes.append(pd.DataFrame({"archivo": d[c_arch], "video": d[c_vid].astype(str).str.slice(0, 24)}))
    if not fuentes:
        return {}
    todo = pd.concat(fuentes, ignore_index=True)
    todo["video"] = todo["video"].astype(str).str.replace(r"\.mp4$", "", regex=True)
    todo["clave"] = todo["archivo"].map(clave_archivo)
    return dict(todo.drop_duplicates("clave").set_index("clave")["video"])


def asignar_video(claves, registro):
    videos, origenes = [], []
    for c in claves:
        if c in registro:
            videos.append(registro[c]); origenes.append("registro del grupo")
        elif (v := video_desde_nombre(c)) is not None:
            videos.append(v); origenes.append("nombre de archivo")
        else:   # último recurso conservador: agrupa de más, nunca genera fugas
            videos.append(re.sub(r"\d+", "", Path(c).stem).strip("_")); origenes.append("prefijo (conservador)")
    return videos, origenes


def huella_dataset(df):
    # Hash de qué imágenes hay y con qué etiquetas: identifica la VERSIÓN del dataset.
    contenido = "\n".join(sorted(df["clave"] + ";" + df["actor"] + ";" + df["emocion"]))
    return hashlib.sha256(contenido.encode()).hexdigest()[:12]


def construir_registro(ruta_dataset, carpeta_curso):
    df = pd.read_csv(ruta_dataset / CSV_MASTER, sep=None, engine="python", encoding="utf-8-sig")
    df.columns = [c.strip().lower() for c in df.columns]
    fisicas = {clave_archivo(p.name): p for p in (ruta_dataset / CARPETA_IMAGENES).iterdir()}
    df = df.assign(clave=df["archivo"].map(clave_archivo), actor=df["actor"].map(normalizar_actor),
                   emocion=df["emocion"].map(normalizar_etiqueta))
    n0 = len(df)
    df = df[df["emocion"].isin(EMOCIONES)].drop_duplicates("clave")
    df = df[df["clave"].isin(fisicas)]
    df = df.assign(ruta=df["clave"].map(lambda c: str(fisicas[c])))
    videos, origenes = asignar_video(df["clave"], leer_registros_grupos(carpeta_curso))
    df["video"] = df["actor"] + "|" + pd.Series(videos, index=df.index).astype(str)   # video calificado con el actor
    df["origen_video"] = origenes
    print(f"filas en el CSV {n0} · imágenes utilizables {len(df)}")
    return df.sort_values("clave").reset_index(drop=True)[["archivo", "clave", "ruta", "actor", "emocion", "video", "origen_video"]]


df = construir_registro(DATASET, CARPETA_CURSO)
HUELLA = huella_dataset(df)
print(f"huella del dataset {HUELLA} · actores {df['actor'].nunique()} · videos {df['video'].nunique()}")
print("origen del video:", df["origen_video"].value_counts().to_dict())
# Sin los registros de los grupos se pierden videos (129 en vez de 154) y la partición cambia: se frena acá.
assert (df["origen_video"] != "nombre de archivo").any(), \
    "No se leyeron los registros de video de los grupos: revisar el acceso directo a la carpeta del curso."

In [ ]:
# 2.2 Distribución actor × emoción
display(pd.crosstab(df["actor"], df["emocion"], margins=True, margins_name="Total"))
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [1, 2]})
df["emocion"].value_counts().reindex(EMOCIONES).plot.bar(ax=axes[0], color="steelblue", title="Imágenes por emoción")
df["actor"].value_counts().plot.bar(ax=axes[1], color="slategray", title="Imágenes por actor")
plt.tight_layout(); plt.show()

**Lectura:** emociones con desbalance moderado (*neutral* la más frecuente, *tristeza* la más escasa) y actores muy desparejos (Francella 419 imágenes, Julio Chávez 71). Por eso se reporta **F1 macro** además de accuracy y se entrena con **pesos por clase** (NB08). Techo realista para emoción: el acuerdo entre etiquetadores de nuestro control cruzado fue **42,6 %**.

## 3. Partición por video
Dos cuadros del mismo plano son casi idénticos: si se reparten entre train y test, el modelo reconoce el plano y no el gesto. Reglas: **ningún video de train aparece en validación ni en test** (se verifica con `assert`); se parte por actor buscando ~70/15/15; si un actor no tiene videos suficientes, val y test comparten **un** video cortado en dos mitades (concesión R4, documentada); train nunca comparte. Es determinista (empates con hash + semilla).

In [ ]:
# 3.1 Partición del dataset
SPLITS = ("train", "val", "test")
MAX_EXHAUSTIVO, MIN_FRAC_EVAL, PENALIDAD_R4 = 16, 0.35, 0.10


def _h(nombre):
    return hashlib.sha256(f"{SEED}:{nombre}".encode()).hexdigest()


def _separable_por_video(tamanios):
    n, total = len(tamanios), tamanios.sum()
    if n < 2:
        return False
    if n > MAX_EXHAUSTIVO:
        return True
    v = tamanios.to_numpy()
    return any(min(s, total - s) >= MIN_FRAC_EVAL * total
               for s in (sum(v[i] for i in range(n) if m >> i & 1) for m in range(1, 2 ** n - 1)))


def _elegir_train(tamanios):
    # Por actor: conjunto de videos cuyo total se acerque más al 70 %, prefiriendo dejar para evaluación
    # videos separables entre val y test (búsqueda exhaustiva con ≤ 16 videos).
    videos = sorted(tamanios.index, key=lambda v: (-tamanios[v], _h(v)))
    total = tamanios.sum(); objetivo = FRACCIONES["train"] * total
    if len(videos) <= MAX_EXHAUSTIVO:
        mejor, mejor_costo = None, np.inf
        for m in range(1, 2 ** len(videos) - 1):
            elegidos = [v for i, v in enumerate(videos) if m >> i & 1]
            costo = abs(tamanios[elegidos].sum() - objetivo) / total
            if not _separable_por_video(tamanios.drop(elegidos)):
                costo += PENALIDAD_R4
            if costo < mejor_costo:
                mejor, mejor_costo = set(elegidos), costo
        return mejor
    elegidos, acum = set(), 0
    for v in videos:
        if acum + tamanios[v] <= objetivo or not elegidos:
            elegidos.add(v); acum += tamanios[v]
    return elegidos - ({videos[-1]} if len(elegidos) == len(videos) else set())


def _repartir_evaluacion(g):
    tamanios = g.groupby("video").size()
    videos = sorted(tamanios.index, key=_h); total = tamanios.sum()
    mejor, mejor_costo = None, np.inf
    if 2 <= len(videos) <= MAX_EXHAUSTIVO:
        for m in range(1, 2 ** len(videos) - 1):
            a_val = [v for i, v in enumerate(videos) if m >> i & 1]
            n_val = tamanios[a_val].sum()
            if min(n_val, total - n_val) >= MIN_FRAC_EVAL * total and abs(n_val - total / 2) < mejor_costo:
                mejor, mejor_costo = set(a_val), abs(n_val - total / 2)
    if mejor is not None:                                     # por video
        return pd.Series(np.where(g["video"].isin(mejor), "val", "test"), index=g.index)
    orden = g.assign(_hv=g["video"].map(_h)).sort_values(["_hv", "clave"]).index   # R4: mitades contiguas
    mitad = len(orden) // 2
    return pd.Series(["val"] * mitad + ["test"] * (len(orden) - mitad), index=orden).reindex(g.index)


def particionar(df):
    split, avisos = pd.Series(index=df.index, dtype=object), []
    for actor, g in df.groupby("actor", sort=True):
        tamanios = g.groupby("video").size()
        if len(tamanios) == 1:                                # un solo video: todo a train
            split.loc[g.index] = "train"; avisos.append(f"{actor}: un solo video -> todo a train"); continue
        es_train = g["video"].isin(_elegir_train(tamanios))
        split.loc[g.index[es_train]] = "train"
        ev = _repartir_evaluacion(g[~es_train]); split.loc[ev.index] = ev
        compartidos = set(g.loc[ev.index[ev == "val"], "video"]) & set(g.loc[ev.index[ev == "test"], "video"])
        if compartidos:
            avisos.append(f"{actor}: val y test comparten {sorted(compartidos)} (R4)")
    return split, avisos


def verificar_sin_fugas(df, split):
    vids = {s: set(df.loc[split == s, "video"]) for s in SPLITS}
    fuga = vids["train"] & (vids["val"] | vids["test"])
    assert not fuga, f"FUGA: {sorted(fuga)[:5]}"
    assert split.notna().all(), "Hay imágenes sin split"
    assert set(df["actor"]) == set(df.loc[split == "train", "actor"]), "Hay actores sin train"


split, avisos = particionar(df)
verificar_sin_fugas(df, split)
assert split.equals(particionar(df)[0]), "La partición no es determinista"
df["split"] = split
print("✔ Sin fugas: ningún video de train aparece en validación o test. ✔ Determinista.")
display(pd.DataFrame({s: {"imágenes": int((split == s).sum()), "videos": df.loc[split == s, "video"].nunique(),
                          "actores": df.loc[split == s, "actor"].nunique()} for s in SPLITS}).T)
print("Concesiones R4:"); [print("  -", a) for a in avisos]

## 4. Baselines (el piso)
Ninguna red se justifica sin un baseline (NB01): el **trivial** (clase más frecuente) y un **Random Forest** sobre los píxeles en gris a 64×64. Se miden en **validación**; el test queda reservado. En el notebook completo está además el MLP y el "acta de límites" (NB02): con los píxeles permutados el MLP rinde igual, prueba de que no usa la estructura espacial y de que hace falta una red convolucional.

In [ ]:
# 4.1 Baselines en validación
def metricas(y_true, y_pred):
    # F1 macro pesa igual a cada clase: con clases desbalanceadas es la métrica honesta.
    return {"accuracy": accuracy_score(y_true, y_pred), "f1_macro": f1_score(y_true, y_pred, average="macro")}


def cargar_pixeles(rutas, tam=IMG_SIZE_BASELINE):
    X = np.empty((len(rutas), tam * tam), dtype=np.float32)
    for i, r in enumerate(rutas):
        with Image.open(r) as im:
            X[i] = np.asarray(im.convert("L").resize((tam, tam), Image.BILINEAR), dtype=np.float32).ravel() / 255.0
    return X


X_pix = cargar_pixeles(df["ruta"])
m_tr, m_va = (df["split"] == "train").to_numpy(), (df["split"] == "val").to_numpy()
resultados_baseline = {}
for tarea in TAREAS:
    filas = []
    for nombre, m in {"Trivial (clase más frecuente)": DummyClassifier(strategy="most_frequent"),
                      "Random Forest (píxeles 64×64)": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                                              n_jobs=-1, random_state=SEED)}.items():
        m.fit(X_pix[m_tr], df.loc[m_tr, tarea])
        filas.append({"modelo": nombre, **metricas(df.loc[m_va, tarea], m.predict(X_pix[m_va]))})
    resultados_baseline[tarea] = pd.DataFrame(filas).set_index("modelo")
    print(f"=== {tarea.upper()} (validación)"); display(resultados_baseline[tarea].round(3))

## 5. Pipeline y herramientas de entrenamiento
Patrón del curso: `map(leer) → cache → shuffle (solo train) → batch → prefetch`. Las imágenes llegan al modelo en [0, 255]; **escalado y aumentación van como capas dentro del modelo** (clase 17), así en inferencia (sección 11) la foto nueva pasa exactamente por el mismo preprocesamiento.

| Aumentación | ¿Válida para rostros? |
|---|---|
| Flip horizontal | Sí: una cara espejada es la misma persona con el mismo gesto |
| Rotación ±14° y zoom ±10 % | Sí: cabezas inclinadas, recortes no centrados |
| Brillo / contraste ±10 % | Sí: fuentes con iluminación distinta |
| Flip vertical, rotaciones grandes | **No**: fabrican caras imposibles (KerasTuner confirmó que la aumentación fuerte empeora) |

**Early stopping sobre el accuracy de validación**, no sobre `val_loss` (D-017): con pocos videos la red se vuelve sobreconfiada y la pérdida de validación sube aunque acierte más; en la primera búsqueda eso llevó a elegir un modelo con 6,5 % de accuracy.

In [ ]:
# 5.1 Datos, aumentación, entrenamiento y evaluación (se usan para la CNN y para ResNet50)
AUTOTUNE = tf.data.AUTOTUNE


def _leer_imagen(ruta, tam):
    img = tf.io.decode_image(tf.io.read_file(ruta), channels=3, expand_animations=False)
    img = tf.cast(tf.round(tf.image.resize(img, (tam, tam))), tf.uint8)
    img.set_shape((tam, tam, 3))
    return img


def datasets_por_split(tarea, tam, batch):
    # Vocabulario de clases fijo (tomado de train); etiquetas enteras; un tf.data por split.
    clases = sorted(df.loc[df["split"] == "train", tarea].unique())
    indice = {c: i for i, c in enumerate(clases)}
    ys, ds = {}, {}
    for s in SPLITS:
        d = df[df["split"] == s]
        ys[s] = d[tarea].map(indice).to_numpy(dtype=np.int32)
        x = tf.data.Dataset.from_tensor_slices((d["ruta"].to_numpy(dtype=str), ys[s]))
        x = x.map(lambda r, y: (_leer_imagen(r, tam), y), num_parallel_calls=AUTOTUNE).cache()
        if s == "train":
            x = x.shuffle(len(d), seed=SEED, reshuffle_each_iteration=True)
        ds[s] = x.map(lambda i, y: (tf.cast(i, tf.float32), y), num_parallel_calls=AUTOTUNE).batch(batch).prefetch(AUTOTUNE)
    return ds, clases, ys


def pesos_por_clase(y, k):
    # Pesos 'balanced' n / (K · n_c) (NB08): un error en una clase chica pesa más.
    conteo = np.bincount(y, minlength=k)
    return {i: float(len(y) / (k * c)) for i, c in enumerate(conteo) if c}


def capas_aumentacion(rotacion, zoom, color, traslacion=0.06):
    # Invarianzas válidas para rostros (NB03, NB16, clase 17). Solo actúan en entrenamiento.
    return keras.Sequential([
        layers.RandomFlip("horizontal", seed=SEED),
        layers.RandomRotation(rotacion, fill_mode="nearest", seed=SEED),
        layers.RandomZoom(zoom, fill_mode="nearest", seed=SEED),
        layers.RandomTranslation(traslacion, traslacion, fill_mode="nearest", seed=SEED),
        layers.RandomBrightness(color, value_range=(0, 255), seed=SEED),
        layers.RandomContrast(color, seed=SEED),
    ], name="aumentacion")


def entrenar(modelo, ds, lr, epocas, paciencia, pesos, desde=0):
    # Adam + early stopping por ACCURACY de validación con los mejores pesos (NB07, NB09, D-017).
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cb = [keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=paciencia,
                                        restore_best_weights=True, start_from_epoch=desde)]
    return modelo.fit(ds["train"], validation_data=ds["val"], epochs=epocas, class_weight=pesos, callbacks=cb, verbose=0)


def evaluar(modelo, ds, ys, split):
    prob = modelo.predict(ds[split], verbose=0)
    return metricas(ys[split], prob.argmax(1)), prob


def graficar_curvas(historias, titulo):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6)); x0 = 0
    for nombre, h in historias.items():
        ep = np.arange(len(h.history["loss"])) + x0
        for ax, met in zip(axes, ("loss", "accuracy")):
            ax.plot(ep, h.history[met], label=f"train {nombre}"); ax.plot(ep, h.history[f"val_{met}"], "--", label=f"val {nombre}")
        x0 += len(h.history["loss"])
    for ax, met in zip(axes, ("pérdida", "accuracy")):
        ax.set_title(f"{titulo} — {met}"); ax.set_xlabel("época"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()


modelos = {}     # (modelo, tarea) -> dict con el modelo, sus datos y resultados

## 6. CNN propia final (desde cero, la que compite)
Arquitectura residual de NB13, **una red por tarea** (una para identidad y otra para emoción), entrada 112×112. Cómo se llegó a esta versión (detalle en `docs/EXPERIMENTOS_CNN.md`):

| Paso | Cambio | Identidad val (acc / F1) | Emoción val (acc / F1) |
|---|---|---|---|
| v1 (control, NB11) | 4 bloques Conv-Conv-BN-Pool, 224×224 | 0,287 / 0,249 | 0,264 / 0,256 |
| v3 · E14 | residual, dos salidas, KerasTuner eligiendo por `val_loss` | 0,252 / 0,224 | 0,290 / 0,270 |
| v3 · E15 | KerasTuner eligiendo por **accuracy** (D-017) | 0,355 / 0,281 | 0,350 / 0,305 |
| **v3 · E16** | búsqueda afinada + **una red por tarea** (D-018) | **0,414 / 0,361** | **0,393 / 0,349** |

| Componente | Dónde se vio | Por qué |
|---|---|---|
| Bloques residuales `x + F(x)` | NB13, clase 7 | Más profundidad sin que el gradiente se pierda |
| Una red por tarea | NB11 (v1), clase 10 | Con dos salidas la emoción competía con la identidad por el mismo tronco (+6 puntos en identidad al separarlas) |
| Entrada 112×112 | decisión de datos | Menos detalle para memorizar el video |
| Aumentación como capas | NB03, NB16, clase 17 | Invarianzas válidas para rostros |
| BatchNorm, Dropout 0,4 | NB07 | Regularización |
| Pesos por clase | NB08 | Desbalance de actores y emociones |
| Adam lr 1,4e-3 + early stopping por accuracy de validación | NB06, NB07, NB09 | Hiperparámetros elegidos por KerasTuner |

In [ ]:
# 6.1 Arquitectura y entrenamiento (una red por tarea)
def bloque_residual(x, filtros, paso, nombre, reg=None):
    # Bloque residual (NB13): la salida es x + F(x); si cambia la forma, el atajo es una conv 1×1.
    atajo = x
    x = layers.Conv2D(filtros, 3, strides=paso, padding="same", use_bias=False, kernel_initializer="he_normal",
                      kernel_regularizer=reg, name=f"{nombre}_conv1")(x)
    x = layers.ReLU()(layers.BatchNormalization(momentum=0.9, name=f"{nombre}_bn1")(x))
    x = layers.Conv2D(filtros, 3, padding="same", use_bias=False, kernel_initializer="he_normal",
                      kernel_regularizer=reg, name=f"{nombre}_conv2")(x)
    x = layers.BatchNormalization(momentum=0.9, name=f"{nombre}_bn2")(x)
    if paso != 1 or atajo.shape[-1] != filtros:
        atajo = layers.Conv2D(filtros, 1, strides=paso, use_bias=False, kernel_regularizer=reg, name=f"{nombre}_atajo")(atajo)
        atajo = layers.BatchNormalization(momentum=0.9, name=f"{nombre}_atajo_bn")(atajo)
    return layers.ReLU()(layers.Add()([x, atajo]))


def cnn_propia(n_clases, nombre, hp=HP_CNN, tam=IMG_SIZE_CNN):
    # Tallo + bloque a resolución completa + 3 bloques que reducen a la mitad y duplican filtros
    # (menos "dónde", más "qué") + GlobalAveragePooling + Dropout + softmax.
    fijar_semillas()
    reg = keras.regularizers.l2(hp["l2"]) if hp["l2"] else None
    base = hp["base"]
    entrada = keras.Input(shape=(tam, tam, 3))
    x = capas_aumentacion(hp["rotacion"], hp["zoom"], hp["color"])(entrada)
    x = layers.Rescaling(1 / 127.5, offset=-1)(x)
    x = layers.Conv2D(base, 3, padding="same", use_bias=False, kernel_initializer="he_normal", kernel_regularizer=reg)(x)
    x = layers.ReLU()(layers.BatchNormalization(momentum=0.9)(x))
    x = bloque_residual(x, base, 1, "r0", reg)
    x = layers.MaxPooling2D()(x)                                 # 112 → 56
    for i, f in enumerate((2 * base, 4 * base, 8 * base), 1):
        x = bloque_residual(x, f, 2, f"r{i}", reg)               # 28 → 14 → 7
    x = layers.Dropout(hp["dropout"])(layers.GlobalAveragePooling2D()(x))
    return keras.Model(entrada, layers.Dense(n_clases, activation="softmax")(x), name=nombre)


for tarea in TAREAS:
    t0 = time.time()
    ds, clases, ys = datasets_por_split(tarea, IMG_SIZE_CNN, BATCH_CNN)
    m = cnn_propia(len(clases), f"cnn_propia_{tarea}")
    h = entrenar(m, ds, HP_CNN["lr"], EPOCAS_MAX_CNN, PACIENCIA_CNN, pesos_por_clase(ys["train"], len(clases)), desde=10)
    val, _ = evaluar(m, ds, ys, "val")
    modelos[("cnn", tarea)] = {"nombre": "CNN propia", "modelo": m, "ds": ds, "clases": clases, "ys": ys, "val": val}
    graficar_curvas({"": h}, f"CNN propia — {tarea}")
    print(f"{tarea}: VAL accuracy={val['accuracy']:.3f} · F1 macro={val['f1_macro']:.3f} · "
          f"{m.count_params():,} parámetros · {len(h.history['loss'])} épocas · {time.time() - t0:.0f} s")

## 7. ResNet50 preentrenada (referencia, no compite)
Los recortes ya son 224×224×3, la entrada nativa de ResNet50, y con ~90 imágenes por actor no alcanza para aprender filtros desde cero. Secuencia del NB15:
- **Fase 1:** base congelada; solo se entrena la cabeza nueva (`GAP → Dropout(0,3) → softmax`) con Adam 1e-3.
- **Fase 2:** se **descongelan** los bloques conv4 y conv5 (vuelven a entrenarse para adaptarse a rostros; las BatchNorm siguen congeladas) y se recompila con Adam **1e-5**, 100 veces menor, para no destruir lo aprendido. En el notebook completo se comparó contra descongelar solo conv5: conv4 + conv5 fue mejor en las dos tareas.

In [ ]:
# 7.1 ResNet50 en dos fases
@keras.saving.register_keras_serializable(package="tp_g6")
class PreprocesoResNet50(layers.Layer):
    # Cada base preentrenada exige SU preprocesamiento (NB13): RGB->BGR y resta de la media de ImageNet.
    def call(self, x):
        return keras.applications.resnet50.preprocess_input(x)


def resnet50_transferencia(n_clases, nombre):
    fijar_semillas()
    base = keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    entrada = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = PreprocesoResNet50()(capas_aumentacion(0.022, 0.10, 0.15, traslacion=0.05)(entrada))
    x = base(x, training=False)          # BatchNorm en modo inferencia aunque luego se descongele
    x = layers.Dropout(0.3)(layers.GlobalAveragePooling2D()(x))
    return keras.Model(entrada, layers.Dense(n_clases, activation="softmax")(x), name=nombre), base


def descongelar_desde(base, primera):
    # Fase 2: desde `primera` en adelante las capas vuelven a entrenarse; las BatchNorm siguen congeladas.
    base.trainable = True
    corte = [l.name for l in base.layers].index(primera)
    for i, capa in enumerate(base.layers):
        capa.trainable = i >= corte and not isinstance(capa, layers.BatchNormalization)


for tarea in TAREAS:
    t0 = time.time()
    ds, clases, ys = datasets_por_split(tarea, IMG_SIZE, BATCH_RESNET)
    pesos = pesos_por_clase(ys["train"], len(clases))
    m, base = resnet50_transferencia(len(clases), f"resnet50_{tarea}")
    h1 = entrenar(m, ds, 1e-3, 20, 5, pesos)
    val1, _ = evaluar(m, ds, ys, "val")
    descongelar_desde(base, RESNET_PRIMERA_DESCONGELADA)        # + recompilar dentro de entrenar()
    h2 = entrenar(m, ds, 1e-5, 20, 5, pesos)
    val, _ = evaluar(m, ds, ys, "val")
    modelos[("resnet", tarea)] = {"nombre": "ResNet50 (conv4+conv5)", "modelo": m, "ds": ds, "clases": clases, "ys": ys,
                                  "val": val, "val fase 1": val1}
    graficar_curvas({"fase 1": h1, "fase 2": h2}, f"ResNet50 — {tarea}")
    print(f"{tarea}: VAL fase 1 acc={val1['accuracy']:.3f} F1={val1['f1_macro']:.3f} → "
          f"fase 2 acc={val['accuracy']:.3f} F1={val['f1_macro']:.3f} · {time.time() - t0:.0f} s")

## 8. Comparación en validación
Acá se toman las decisiones. La tabla por actor muestra **para quién funciona** cada modelo y conecta con el desbalance: el profesor habilitó excluir actores que ningún modelo reconoce, pero el único caso (Eva De Dominici) tiene 8 videos, no le faltan datos, así que **no se excluyó a nadie** (D-018).

In [ ]:
# 8.1 Tabla de validación y recall por actor
filas = []
for tarea in TAREAS:
    for nombre, fila in resultados_baseline[tarea].iterrows():
        filas.append({"tarea": tarea, "modelo": nombre, **fila.to_dict()})
    filas.append({"tarea": tarea, "modelo": "CNN propia (desde cero)", **modelos[("cnn", tarea)]["val"]})
    filas.append({"tarea": tarea, "modelo": "ResNet50 fase 1 (base congelada)", **modelos[("resnet", tarea)]["val fase 1"]})
    filas.append({"tarea": tarea, "modelo": "ResNet50 fase 2 (conv4+conv5)", **modelos[("resnet", tarea)]["val"]})
display(pd.DataFrame(filas).set_index(["tarea", "modelo"]).style.format("{:.3f}"))


def recall_por_clase(r, split="val"):
    pred = r["modelo"].predict(r["ds"][split], verbose=0).argmax(1)
    cm = confusion_matrix(r["ys"][split], pred, labels=range(len(r["clases"])))
    return pd.Series(np.diag(cm) / np.maximum(cm.sum(1), 1), index=r["clases"])


val = df[df["split"] == "val"]
display(pd.DataFrame({"CNN propia · recall": recall_por_clase(modelos[("cnn", "actor")]),
                      "ResNet50 · recall": recall_por_clase(modelos[("resnet", "actor")]),
                      "imágenes train": df.loc[df["split"] == "train", "actor"].value_counts(),
                      "imágenes val": val["actor"].value_counts(),
                      "videos": df.groupby("actor")["video"].nunique()})
        .sort_values("CNN propia · recall").style.format({"CNN propia · recall": "{:.2f}", "ResNet50 · recall": "{:.2f}"}))

## 9. Test — una sola vez
Todas las decisiones se tomaron con validación. Ahora se evalúan en test los modelos finales; esta celda no se vuelve a correr para "probar otra cosa".

In [ ]:
# 9.1 Resultados en test y matrices de confusión
resultados_test = {}
for clave, r in modelos.items():
    met, prob = evaluar(r["modelo"], r["ds"], r["ys"], "test")
    resultados_test[clave] = {"metricas": met, "prob": prob}
display(pd.DataFrame([{"tarea": t, "modelo": modelos[(m, t)]["nombre"], **resultados_test[(m, t)]["metricas"]}
                      for (m, t) in modelos]).set_index(["tarea", "modelo"]).style.format("{:.3f}"))
print(f"Huella del dataset: {HUELLA} · semilla: {SEED}")

for clave in [("cnn", "actor"), ("cnn", "emocion"), ("resnet", "actor"), ("resnet", "emocion")]:
    r, pred = modelos[clave], resultados_test[clave]["prob"].argmax(1)
    print(f"\n===== {r['nombre']} · {clave[1]} (test) =====")
    print(classification_report(r["ys"]["test"], pred, labels=range(len(r["clases"])), target_names=r["clases"], zero_division=0))
    lado = max(5, 0.45 * len(r["clases"]) + 2)
    fig, ax = plt.subplots(figsize=(lado, lado))
    cm = confusion_matrix(r["ys"]["test"], pred, labels=range(len(r["clases"])), normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=r["clases"]).plot(ax=ax, cmap="Blues", colorbar=False, values_format=".2f", xticks_rotation=90)
    ax.set_title(f"{r['nombre']} — {clave[1]} (test, normalizada por fila)"); plt.tight_layout(); plt.show()

## 10. ¿El modelo mira la cara? (oclusión, NB12)
Se tapa cada región con un parche gris y se mide cuánto cae la probabilidad de la clase correcta: en rojo, las regiones decisivas. Es la prueba del riesgo declarado en el TP1: como cada actor sale de pocos videos, un modelo podría reconocer **el fondo o la serie** en lugar de la cara.

In [ ]:
# 10.1 Mapas de oclusión sobre aciertos de test
def mapa_oclusion(modelo, imagen, clase, gris=127.5):
    lado = imagen.shape[0]; parche, paso = lado // 7, lado // 14
    pos = list(range(0, lado - parche + 1, paso)); lote = []
    for y in pos:
        for x in pos:
            t = imagen.copy(); t[y:y + parche, x:x + parche] = gris; lote.append(t)
    p0 = float(modelo.predict(imagen[None], verbose=0)[0, clase])
    caidas = (p0 - modelo.predict(np.stack(lote), batch_size=64, verbose=0)[:, clase]).reshape(len(pos), len(pos))
    mapa, cuenta = np.zeros((lado, lado)), np.zeros((lado, lado))
    for a, y in enumerate(pos):
        for b, x in enumerate(pos):
            mapa[y:y + parche, x:x + parche] += caidas[a, b]; cuenta[y:y + parche, x:x + parche] += 1
    return mapa / np.maximum(cuenta, 1), p0


rutas_test = df.loc[df["split"] == "test", "ruta"].to_numpy()
for clave in [("cnn", "actor"), ("cnn", "emocion"), ("resnet", "actor"), ("resnet", "emocion")]:
    r, prob = modelos[clave], resultados_test[clave]["prob"]
    lado = r["modelo"].inputs[0].shape[1]                       # CNN 112 · ResNet50 224
    bien = np.where(prob.argmax(1) == r["ys"]["test"])[0]
    elegidas = np.random.default_rng(SEED).choice(bien, size=min(5, len(bien)), replace=False)
    fig, axes = plt.subplots(2, len(elegidas), figsize=(3 * len(elegidas), 6.2))
    for k, i in enumerate(elegidas):
        img = np.asarray(Image.open(rutas_test[i]).convert("RGB").resize((lado, lado), Image.BILINEAR), dtype="float32")
        mapa, p = mapa_oclusion(r["modelo"], img, int(r["ys"]["test"][i]))
        axes[0, k].imshow(img.astype("uint8")); axes[0, k].set_title(f"{r['clases'][r['ys']['test'][i]]} (p={p:.2f})", fontsize=9)
        axes[1, k].imshow(img.astype("uint8")); axes[1, k].imshow(mapa, cmap="jet", alpha=0.5)
        axes[0, k].axis("off"); axes[1, k].axis("off")
    fig.suptitle(f"Oclusión — {r['nombre']} — {clave[1]} (rojo = región decisiva)"); plt.tight_layout(); plt.show()

## 11. Predicción sobre fotos nuevas (competencia)
Se sube cualquier foto. Como el modelo se entrenó con **recortes de rostro**, la foto se recorta a un cuadrado central y se reduce al tamaño de cada red; **conviene subir la foto recortada a la cara** (sin detector de rostros, que no se vio en la cátedra). La CNN propia es la que compite; ResNet50 se muestra como referencia. Para cada tarea se muestran las 3 clases más probables.

In [ ]:
# 11.1 Subir fotos y predecir actor y emoción
from google.colab import files


def preparar_foto(ruta, lado):
    # Corrige la orientación EXIF del celular, recorta el cuadrado central y lo lleva al tamaño de la red.
    img = ImageOps.exif_transpose(Image.open(ruta)).convert("RGB")
    w, h = img.size; s = min(w, h)
    img = img.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
    return np.asarray(img.resize((lado, lado), Image.BILINEAR), dtype="float32")


def predecir(ruta, familia="cnn", top=3):
    salida = {}
    for tarea in TAREAS:
        r = modelos[(familia, tarea)]
        prob = r["modelo"].predict(preparar_foto(ruta, r["modelo"].inputs[0].shape[1])[None], verbose=0)[0]
        salida[tarea] = [(r["clases"][i], float(prob[i])) for i in np.argsort(-prob)[:top]]
    return salida


subidas = files.upload()
for nombre in subidas:
    cnn, res = predecir(nombre, "cnn"), predecir(nombre, "resnet")
    plt.figure(figsize=(3.5, 3.5)); plt.imshow(preparar_foto(nombre, IMG_SIZE).astype("uint8")); plt.axis("off")
    plt.title(f"CNN propia: {cnn['actor'][0][0]} · {cnn['emocion'][0][0]}", fontsize=10); plt.show()
    for familia, p in (("CNN propia (compite)", cnn), ("ResNet50 (referencia)", res)):
        print(f"  {familia}")
        for tarea in TAREAS:
            print(f"    {tarea:8s}: " + " · ".join(f"{c} {q:.0%}" for c, q in p[tarea]))

In [ ]:
# 11.2 (Opcional) Guardar las dos CNN entrenadas para predecir en otra sesión sin reentrenar
# Se guardan en el disco temporal de Colab y se descargan a la computadora (no se suben al repo ni al Drive del curso).
Path("/content/modelos").mkdir(exist_ok=True)
for tarea in TAREAS:
    destino = f"/content/modelos/cnn_propia_{tarea}.keras"
    modelos[("cnn", tarea)]["modelo"].save(destino)
    files.download(destino)
# Para recargarlas: keras.models.load_model("cnn_propia_actor.keras") y usar las clases de `modelos[("cnn", "actor")]["clases"]`
# (orden alfabético de los actores de train).

## 12. Conclusiones
Valores de la corrida registrada del 29/10 (E16); al volver a correr pueden variar levemente por la GPU.

| Modelo | Identidad val | Emoción val | Identidad test | Emoción test |
|---|---|---|---|---|
| Random Forest (baseline) | 0,334 / 0,277 | 0,255 / 0,167 | — | — |
| **CNN propia final** | **0,414 / 0,361** | **0,393 / 0,349** | **0,352 / 0,328** | **0,298 / 0,290** |
| ResNet50 conv4+conv5 | 0,654 / 0,550 | 0,477 / 0,455 | 0,660 / 0,547 | 0,362 / 0,341 |

(accuracy / F1 macro)

1. **El dato manda más que la arquitectura.** Con 20 actores de 2 a 25 videos cada uno, la CNN desde cero memoriza el video (≈ 90 % en train). La partición por video es lo que hace honestos estos números: con una partición por imagen el resultado se infla.
2. **La CNN propia supera a los baselines** (+8 puntos de accuracy en identidad y +14 en emoción sobre el Random Forest), y la mejora vino de decisiones de la cátedra: bloques residuales, regularización, aumentación suave, KerasTuner y separar las tareas.
3. **Elegir por accuracy y no por pérdida** fue el cambio más importante: con pocos datos la red se vuelve sobreconfiada y `val_loss` deja de reflejar los aciertos.
4. **La transferencia (ResNet50) casi duplica a la CNN en identidad:** los filtros aprendidos en ImageNet compensan la falta de datos. En emoción la ventaja es menor: es una tarea difícil incluso para personas (acuerdo entre etiquetadores 42,6 %).
5. **Desbalance:** los pesos por clase ayudan, pero hay actores que ningún modelo reconoce bien (Eva De Dominici, Colapinto) y confusiones razonables entre parecidos (Tini ↔ Lali, Darín ↔ Echarri). No se excluyó a ningún actor.
6. **Limitaciones:** pocos videos por actor, R4 en 9 actores (val y test comparten un video), fotos de la competencia distintas a los recortes del curso (por eso conviene recortar la cara) y caída de validación a test por elegir entre muchas variantes con 428 imágenes de validación.

**Declaración de uso de IA generativa:** se usó un asistente de IA (Claude) para auditar el dataset, proponer la estructura del código y redactar borradores de la documentación. Todas las decisiones fueron revisadas por el grupo.